# Phase 3 — Audio Bi-LSTM (DAIC-WOZ)

**Thesis:** Multimodal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University

This notebook trains the **Audio Branch** — a Bidirectional LSTM on MFCC features extracted from DAIC-WOZ interview recordings.

**Why MFCC + delta + delta2?**
- MFCC captures the spectral shape of speech (timbre, vowel quality)
- Delta captures rate of change (speech tempo, pausing)
- Delta-delta captures acceleration (hesitation, irregular rhythm)
- Together they encode the acoustic patterns that correlate with depression

**Why Bi-LSTM?**  
Depression manifests in temporal patterns — long pauses, falling intonation, slow speech. Bidirectional context captures these better than feed-forward models.

## 1. Setup

In [ ]:
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. DAIC-WOZ Audio Data

Expected structure after download:
```
data/raw/daic/
├── train_split_Depression_AVEC2017.csv
├── dev_split_Depression_AVEC2017.csv
├── 300_P/300_AUDIO.wav
├── 301_P/301_AUDIO.wav
└── ...
```
Without DAIC-WOZ, synthetic MFCC data is generated automatically.

In [ ]:
# Optional: load DAIC-WOZ from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/daic depression_thesis/data/raw/daic

import os
print('DAIC data found:', os.path.exists('data/raw/daic'))

## 3. Extract MFCC Features

In [ ]:
!python src/preprocessing/audio_preprocess.py

In [ ]:
import numpy as np

X_train = np.load('data/processed/daic_audio/X_train.npy')
y_train = np.load('data/processed/daic_audio/y_train.npy')
X_dev   = np.load('data/processed/daic_audio/X_dev.npy')
y_dev   = np.load('data/processed/daic_audio/y_dev.npy')

print(f'Train: {X_train.shape}  |  Dev: {X_dev.shape}')
print(f'Shape: (samples, time_steps, features)')
print(f'Train — Not depressed: {(y_train==0).sum()}  |  Depressed: {(y_train==1).sum()}')

## 4. Visualize MFCC Features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
labels_text = {0: 'Not Depressed', 1: 'Depressed'}
colors = {0: 'steelblue', 1: 'tomato'}

for cls in [0, 1]:
    idx = np.where(y_train == cls)[0][0]
    mfcc_only = X_train[idx, :, :40].T  # first 40 dims = MFCC
    axes[cls].imshow(mfcc_only, aspect='auto', origin='lower', cmap='coolwarm')
    axes[cls].set_title(f'MFCC Spectrogram — {labels_text[cls]}',
                        color=colors[cls], fontsize=11)
    axes[cls].set_xlabel('Time steps'); axes[cls].set_ylabel('MFCC coeff')

plt.tight_layout()
plt.savefig('results/figures/phase3_mfcc_samples.png', dpi=150)
plt.show()

## 5. Train Bi-LSTM

In [ ]:
!python src/training/train_audio.py

## 6. Detailed Evaluation

In [ ]:
import sys, torch, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
sys.path.insert(0, '.')
from src.models.audio_lstm import AudioLSTM
from torch.utils.data import TensorDataset, DataLoader
from configs.config import N_MFCC

N_FEAT = N_MFCC * 3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = AudioLSTM(n_mfcc=N_FEAT, hidden=128, num_classes=2).to(device)
model.load_state_dict(torch.load('results/audio_lstm_phase3.pth', map_location=device))
model.eval()

dev_dl = DataLoader(TensorDataset(torch.tensor(X_dev), torch.tensor(y_dev)), batch_size=64)

all_preds, all_probs, all_true = [], [], []
with torch.no_grad():
    for xb, yb in dev_dl:
        logits = model(xb.to(device))
        all_probs.extend(torch.softmax(logits,1)[:,1].cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

all_true = np.array(all_true); all_preds = np.array(all_preds); all_probs = np.array(all_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', ax=ax1,
            xticklabels=['Not Dep','Depressed'],
            yticklabels=['Not Dep','Depressed'], cmap='Reds')
ax1.set_title('Confusion Matrix — Audio Bi-LSTM'); ax1.set_ylabel('True'); ax1.set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(all_true, all_probs)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='tomato', lw=2, label=f'AUC = {roc_auc:.3f}')
ax2.plot([0,1],[0,1],'k--'); ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('ROC Curve — Audio Bi-LSTM'); ax2.legend()

plt.tight_layout()
plt.savefig('results/figures/phase3_confusion_roc.png', dpi=150)
plt.show()

print(classification_report(all_true, all_preds, target_names=['Not Depressed','Depressed']))

with open('results/metrics/phase3_metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))